# 10｜SIRIUS 6.5.4 概率指纹字典与矩阵提取

本 Notebook 将正、负模式拆成两个可独立运行的模块。每次只读取 **当时已经在 SIRIUS GUI 中打开的一个模式项目**：

1. 连接本机正在运行的 SIRIUS 6 REST 服务；
2. 先在 GUI 中打开负模式项目，运行“模块一：负模式”；
3. 再在 GUI 中打开正模式项目，运行“模块二：正模式”；
4. 对每个 feature 选择 **SIRIUS rank=1 分子式**；
5. 提取与该分子式对应的 5,000 维概率指纹；
6. 按 `relativeIndex` 验证向量顺序，按 `absoluteIndex` 对齐正负模式定义；
7. 保存矩阵、行索引、失败原因、QC和运行清单。

本步骤不进行结构数据库搜索，不读取 ChemVista 谱图或外部标签，也不根据正确结构挑选分子式。

## 运行前准备

1. 启动 SIRIUS GUI 并保持窗口开启。先手工打开 `negative_ms.sirius`。
2. 运行公共准备单元和负模式模块；负模式结果保存完成后，再在 GUI 中打开 `positive_ms.sirius`。
3. 在 PyCharm 中选择 `Python (env_ms)` / `D:\Appinstallation\Anaconda\envs\env_ms\python.exe`。
4. 运行正模式模块。两个模式都完成后，再运行最下方的汇总QC与清单单元。

> 主分析严格使用 rank=1 分子式。若 rank=1 没有概率指纹，该 feature 记录为失败，不自动改用低排名分子式。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import sys
import time
import urllib.error
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').exists(), f'未找到项目根目录，当前目录：{Path.cwd()}'

SIRIUS_INPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'sirius_v1' / 'development_inputs_v1'
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'sirius_v1' / 'development_results_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_OVERWRITE_COMPLETED_EXPORT = True
COMPLETED_MANIFEST_PATH = OUTPUT_DIR / 'development_sirius_result_manifest.json'
if COMPLETED_MANIFEST_PATH.exists() and not ALLOW_OVERWRITE_COMPLETED_EXPORT:
    raise RuntimeError(
        f'检测到已经完成的正式导出：{COMPLETED_MANIFEST_PATH}。'
        '为避免覆盖冻结结果，本次停止。确需重跑时请先建立新版本目录，'
        '或明确把 ALLOW_OVERWRITE_COMPLETED_EXPORT 改为 True。'
    )

FEATURE_MANIFEST_PATH = SIRIUS_INPUT_DIR / 'development_sirius_feature_manifest.csv.gz'
RETRY_AUDIT_PATH = SIRIUS_INPUT_DIR / 'sirius_uniform_retry_audit_v1.json'
PROJECTS = {
    'negative': {
        'charge': -1,
        'project_path': SIRIUS_INPUT_DIR / 'negative_ms.sirius',
        'expected_features': 601,
        'project_id': 'pbt_dev_negative',
    },
    'positive': {
        'charge': 1,
        'project_path': SIRIUS_INPUT_DIR / 'positive_ms.sirius',
        'expected_features': 1638,
        'project_id': 'pbt_dev_positive',
    },
}

assert FEATURE_MANIFEST_PATH.is_file(), FEATURE_MANIFEST_PATH
assert RETRY_AUDIT_PATH.is_file(), RETRY_AUDIT_PATH
RETRY_AUDIT = json.loads(RETRY_AUDIT_PATH.read_text(encoding='utf-8'))
assert RETRY_AUDIT.get('status') == 'COMPLETED_REPRODUCIBLE_404', RETRY_AUDIT

for mode, cfg in PROJECTS.items():
    assert cfg['project_path'].is_file(), f"缺少 {mode} 项目：{cfg['project_path']}"
    audited = RETRY_AUDIT['projects'][mode]
    current_size = int(cfg['project_path'].stat().st_size)
    audited_size = int(audited['after_retry_size_bytes'])
    if current_size != audited_size:
        print(
            f'提示：{mode}项目容器大小在SIRIUS保存后发生变化：'
            f'{audited_size} -> {current_size} bytes；最终清单将记录当前文件身份。'
        )
print('Python:', sys.executable)
print('Project root:', ROOT)
print('Output:', OUTPUT_DIR)

In [ ]:
# SIRIUS GUI 会把本地 REST 端口写入该文件。
SIRIUS_PORT_FILE = Path.home() / '.sirius' / 'sirius-6.port'

def read_sirius_base_url():
    if not SIRIUS_PORT_FILE.is_file():
        raise RuntimeError('未找到 SIRIUS 端口文件。请先启动 SIRIUS GUI，并保持窗口开启。')
    port = int(SIRIUS_PORT_FILE.read_text(encoding='utf-8').strip())
    return f'http://127.0.0.1:{port}'

BASE_URL = read_sirius_base_url()

def api_request(method, path, params=None, expect='json', timeout=180, retries=3):
    query = urllib.parse.urlencode(params or {}, doseq=True)
    url = BASE_URL + path + (('?' + query) if query else '')
    data = b'' if method.upper() in {'PUT', 'POST'} else None
    request = urllib.request.Request(url, data=data, method=method.upper())
    request.add_header('Accept', 'application/json, application/csv, text/plain')
    last_error = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                raw = response.read()
            if expect == 'bytes':
                return raw
            text = raw.decode('utf-8-sig')
            if expect == 'text':
                return text
            return json.loads(text) if text.strip() else None
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError) as exc:
            last_error = exc
            if isinstance(exc, urllib.error.HTTPError):
                detail = exc.read().decode('utf-8', errors='replace')
                # 404 表示该 feature/formula 没有对应结果，不重复请求。
                if exc.code == 404:
                    raise RuntimeError(f'HTTP 404: {detail}') from exc
            if attempt + 1 < retries:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'SIRIUS API 请求失败：{method} {url}；{last_error}') from last_error

def normalize_windows_path(value):
    return str(Path(value).resolve()).rstrip('\\/').casefold()

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

print('SIRIUS REST:', BASE_URL)

In [ ]:
# 检查服务和版本。主分析要求 SIRIUS 6；本项目当前运行版本应为 6.5.4。
SIRIUS_INFO = api_request(
    'GET', '/api/info',
    params={'includeConsumables': 'false', 'includeVersions': 'true'},
)
display(pd.Series(SIRIUS_INFO, name='value').to_frame())
assert str(SIRIUS_INFO.get('siriusVersion', '')).startswith('6.'), SIRIUS_INFO
if SIRIUS_INFO.get('siriusVersion') != '6.5.4':
    print('警告：当前版本不是本项目已记录的 6.5.4，请把实际版本写入运行清单并单独版本化结果。')

In [ ]:
feature_manifest = pd.read_csv(FEATURE_MANIFEST_PATH)
assert feature_manifest['sirius_feature_id'].is_unique
feature_manifest['_mode_key'] = feature_manifest['ion_mode'].astype(str).str.casefold()

def get_project_features(project_id):
    records = api_request(
        'GET',
        f'/api/projects/{urllib.parse.quote(project_id, safe="")}/aligned-features',
        params={'includeMergedMs2': 'false'},
    )
    features = pd.DataFrame(records)
    if features.empty:
        return features
    features['sirius_feature_id'] = features['externalFeatureId'].fillna(features['name'])
    return features.sort_values('sirius_feature_id').reset_index(drop=True)

def find_current_project_for_mode(mode):
    # 只查找 GUI 当前已经打开的项目；不会自动打开另一个正/负模式项目。
    cfg = PROJECTS[mode]
    expected_ids = set(
        feature_manifest.loc[feature_manifest['_mode_key'].eq(mode.casefold()), 'sirius_feature_id']
    )
    assert len(expected_ids) == cfg['expected_features'], (mode, len(expected_ids), cfg['expected_features'])

    exact_matches = []
    inspected = []
    for project in api_request('GET', '/api/projects'):
        project_id = project['projectId']
        try:
            features = get_project_features(project_id)
        except Exception as exc:
            inspected.append((project_id, 'unreadable', str(exc)[:160]))
            continue
        observed_ids = set(features['sirius_feature_id']) if not features.empty else set()
        inspected.append((project_id, len(features), project.get('location')))
        if observed_ids == expected_ids:
            exact_matches.append((project_id, features, project.get('location')))

    if len(exact_matches) == 1:
        project_id, features, location = exact_matches[0]
        print(f'已识别 {mode} 项目：{project_id}；features={len(features)}；location={location}')
        return project_id, features
    if len(exact_matches) > 1:
        target = normalize_windows_path(cfg['project_path'])
        path_matches = [item for item in exact_matches if item[2] and normalize_windows_path(item[2]) == target]
        if len(path_matches) == 1:
            project_id, features, location = path_matches[0]
            print(f'已按保存路径识别 {mode} 项目：{project_id}；features={len(features)}')
            return project_id, features
        raise RuntimeError(f'发现多个内容完全相同的 {mode} 项目，请在 SIRIUS 中只保留一个：{[(x[0], x[2]) for x in exact_matches]}')

    raise RuntimeError(
        f'当前 SIRIUS GUI 中没有找到完整的 {mode} 项目。'
        f'请先在 SIRIUS 中打开 {cfg["project_path"]}，再运行本模块。'
        f'当前检查到的项目：{inspected}'
    )

In [ ]:
def load_fingerprint_dictionary(project_id, mode, charge):
    text = api_request(
        'GET',
        f'/api/projects/{urllib.parse.quote(project_id, safe="")}/fingerid-data',
        params={'charge': charge},
        expect='text',
    )
    # 接口声明为 CSV，但 SIRIUS 6.5.4 实际返回制表符分隔文本。
    dictionary = pd.read_csv(io.StringIO(text), sep='\t')
    dictionary = dictionary.sort_values('relativeIndex').reset_index(drop=True)
    dictionary.insert(0, 'mode', mode)
    dictionary.insert(1, 'charge', charge)
    dictionary['sirius_version'] = SIRIUS_INFO.get('siriusVersion')
    dictionary['fingerprint_id'] = SIRIUS_INFO.get('fingerprintId')
    dictionary['column_name'] = [
        f'{mode}_abs_{int(idx):06d}' for idx in dictionary['absoluteIndex']
    ]
    assert dictionary['relativeIndex'].tolist() == list(range(len(dictionary)))
    assert dictionary['absoluteIndex'].is_unique
    return dictionary

def save_mode_results(mode, dictionary, matrix, status):
    dictionary_path = OUTPUT_DIR / f'fingerprint_dictionary_{mode}.csv'
    matrix_path = OUTPUT_DIR / f'{mode}_fingerprint_probabilities.npy'
    rows_path = OUTPUT_DIR / f'{mode}_fingerprint_rows.csv.gz'
    dictionary.to_csv(dictionary_path, index=False, encoding='utf-8-sig')
    np.save(matrix_path, matrix, allow_pickle=False)
    status.to_csv(rows_path, index=False, compression='gzip')
    print('dictionary:', dictionary_path)
    print('matrix:', matrix_path, matrix.shape, matrix.dtype)
    print('rows/status:', rows_path)
    print(status['status'].value_counts(dropna=False).to_string())

In [ ]:
def extract_mode_fingerprints(mode, project_id, features, dictionary):
    dimension = len(dictionary)
    matrix = np.full((len(features), dimension), np.nan, dtype=np.float32)
    status_rows = []
    project_key = urllib.parse.quote(project_id, safe='')

    for matrix_row, feature in tqdm(
        features.iterrows(), total=len(features), desc=f'提取 {mode} 指纹'
    ):
        feature_id = str(feature['alignedFeatureId'])
        external_id = str(feature['sirius_feature_id'])
        base_status = {
            'matrix_row': int(matrix_row),
            'mode': mode,
            'charge': int(feature['charge']),
            'sirius_feature_id': external_id,
            'aligned_feature_id': feature_id,
            'ion_mass': feature.get('ionMass'),
            'has_msms': feature.get('hasMsMs'),
            'formula_id': None,
            'molecular_formula': None,
            'adduct': None,
            'formula_rank': None,
            'sirius_score': None,
            'sirius_score_normalized': None,
            'fingerprint_length': 0,
            'status': None,
            'error': None,
        }
        try:
            feature_key = urllib.parse.quote(feature_id, safe='')
            formulas = api_request(
                'GET',
                f'/api/projects/{project_key}/aligned-features/{feature_key}/formulas',
            )
            if not formulas:
                base_status['status'] = 'no_formula_candidates'
                status_rows.append(base_status)
                continue

            rank1 = [item for item in formulas if item.get('rank') == 1]
            if not rank1:
                base_status['status'] = 'no_rank1_formula'
                status_rows.append(base_status)
                continue
            formula = rank1[0]
            formula_id = str(formula['formulaId'])
            base_status.update({
                'formula_id': formula_id,
                'molecular_formula': formula.get('molecularFormula'),
                'adduct': formula.get('adduct'),
                'formula_rank': formula.get('rank'),
                'sirius_score': formula.get('siriusScore'),
                'sirius_score_normalized': formula.get('siriusScoreNormalized'),
            })
            formula_key = urllib.parse.quote(formula_id, safe='')
            probabilities = api_request(
                'GET',
                f'/api/projects/{project_key}/aligned-features/{feature_key}/formulas/{formula_key}/fingerprint',
            )
            probabilities = np.asarray(probabilities, dtype=np.float32)
            base_status['fingerprint_length'] = int(probabilities.size)
            if probabilities.size != dimension:
                base_status['status'] = 'fingerprint_length_mismatch'
                base_status['error'] = f'expected={dimension}; observed={probabilities.size}'
                status_rows.append(base_status)
                continue
            if not np.isfinite(probabilities).all():
                base_status['status'] = 'fingerprint_nonfinite'
                status_rows.append(base_status)
                continue
            if ((probabilities < 0) | (probabilities > 1)).any():
                base_status['status'] = 'fingerprint_out_of_range'
                status_rows.append(base_status)
                continue

            matrix[matrix_row] = probabilities
            base_status['status'] = 'success'
        except Exception as exc:
            error_text = str(exc)[:2000]
            retry_confirmed_404 = (
                RETRY_AUDIT.get('status') == 'COMPLETED_REPRODUCIBLE_404'
                and error_text.startswith('HTTP 404:')
                and 'Fingerprint for' in error_text
                and 'not found' in error_text
            )
            base_status['status'] = (
                'fingerprint_unavailable_after_retry'
                if retry_confirmed_404 else 'fingerprint_request_failed'
            )
            base_status['error'] = error_text
        status_rows.append(base_status)

    status = pd.DataFrame(status_rows).sort_values('matrix_row').reset_index(drop=True)
    assert status['matrix_row'].tolist() == list(range(len(features)))
    assert status['sirius_feature_id'].is_unique
    return matrix, status


## 模块一｜负模式项目提取

先在 SIRIUS GUI 中打开 `negative_ms.sirius`，确认项目显示 601 个 feature，并保持软件开启。然后只运行下面这个负模式代码单元。完成后会立即保存负模式字典、矩阵和状态表。

In [ ]:
# 模块一：先在 SIRIUS GUI 中打开 negative_ms.sirius，再运行本单元。
mode = 'negative'
project_id, features = find_current_project_for_mode(mode)
dictionary = load_fingerprint_dictionary(project_id, mode, PROJECTS[mode]['charge'])
matrix, status = extract_mode_fingerprints(mode, project_id, features, dictionary)
save_mode_results(mode, dictionary, matrix, status)
display(dictionary.head(5))

## 模块二｜正模式项目提取

负模式结果保存后，在 SIRIUS GUI 中打开 `positive_ms.sirius`，确认项目显示 1,638 个 feature。随后运行下面这个正模式代码单元。负模式项目不需要同时保持打开。

In [ ]:
# 模块二：在 SIRIUS GUI 中切换到 positive_ms.sirius 后，再运行本单元。
mode = 'positive'
project_id, features = find_current_project_for_mode(mode)
dictionary = load_fingerprint_dictionary(project_id, mode, PROJECTS[mode]['charge'])
matrix, status = extract_mode_fingerprints(mode, project_id, features, dictionary)
save_mode_results(mode, dictionary, matrix, status)
display(dictionary.head(5))

## 两个模式完成后｜跨模式对齐、QC与运行清单

以下单元只读取已经保存到磁盘的正负模式结果，因此不要求两个 `.sirius` 项目同时在GUI中打开。只有在模块一和模块二均成功保存后再运行。

In [ ]:
# 从磁盘重新读取两个模块的结果；即使中间切换项目或重启kernel也可以继续。
required_mode_outputs = []
for mode in ('positive', 'negative'):
    required_mode_outputs.extend([
        OUTPUT_DIR / f'fingerprint_dictionary_{mode}.csv',
        OUTPUT_DIR / f'{mode}_fingerprint_probabilities.npy',
        OUTPUT_DIR / f'{mode}_fingerprint_rows.csv.gz',
    ])
missing_outputs = [str(path) for path in required_mode_outputs if not path.is_file()]
assert not missing_outputs, f'请先完成负模式和正模式两个模块。缺少：{missing_outputs}'

FINGERPRINT_DICTIONARIES = {
    mode: pd.read_csv(OUTPUT_DIR / f'fingerprint_dictionary_{mode}.csv')
    for mode in ('positive', 'negative')
}
MATRICES = {
    mode: np.load(OUTPUT_DIR / f'{mode}_fingerprint_probabilities.npy', allow_pickle=False)
    for mode in ('positive', 'negative')
}
STATUS_TABLES = {
    mode: pd.read_csv(OUTPUT_DIR / f'{mode}_fingerprint_rows.csv.gz')
    for mode in ('positive', 'negative')
}

# 使用 absoluteIndex 建立正负模式位定义对照；不能按相同 relativeIndex 盲目合并。
pos_dict = FINGERPRINT_DICTIONARIES['positive'].copy()
neg_dict = FINGERPRINT_DICTIONARIES['negative'].copy()
alignment = pos_dict.merge(
    neg_dict, on='absoluteIndex', how='outer', suffixes=('_positive', '_negative'), indicator=True
)
alignment['description_consistent'] = (
    alignment['description_positive'].fillna('') == alignment['description_negative'].fillna('')
)
alignment['type_consistent'] = (
    alignment['Type_positive'].fillna('') == alignment['Type_negative'].fillna('')
)
alignment_path = OUTPUT_DIR / 'fingerprint_cross_mode_alignment.csv.gz'
alignment.to_csv(alignment_path, index=False, compression='gzip')
display(alignment['_merge'].value_counts(dropna=False).rename('n').to_frame())
print('saved:', alignment_path)

In [ ]:
qc_rows = []

def add_qc(check, passed, observed, expected, severity='FAIL'):
    qc_rows.append({
        'check': check,
        'status': 'PASS' if passed else severity,
        'observed': observed,
        'expected': expected,
    })

add_qc('sirius_major_version_6', str(SIRIUS_INFO.get('siriusVersion', '')).startswith('6.'),
       SIRIUS_INFO.get('siriusVersion'), '6.x')

for mode, cfg in PROJECTS.items():
    dictionary = FINGERPRINT_DICTIONARIES[mode]
    matrix = MATRICES[mode]
    status = STATUS_TABLES[mode]
    success_mask = status['status'].eq('success').to_numpy()

    add_qc(f'{mode}_feature_count', len(status) == cfg['expected_features'],
           len(status), cfg['expected_features'])
    add_qc(f'{mode}_dictionary_relative_index_contiguous',
           dictionary['relativeIndex'].tolist() == list(range(len(dictionary))),
           len(dictionary), '0..dimension-1')
    add_qc(f'{mode}_dictionary_absolute_index_unique', dictionary['absoluteIndex'].is_unique,
           dictionary['absoluteIndex'].nunique(), len(dictionary))
    add_qc(f'{mode}_matrix_shape', matrix.shape == (len(status), len(dictionary)),
           str(matrix.shape), str((len(status), len(dictionary))))
    add_qc(f'{mode}_status_feature_id_unique', status['sirius_feature_id'].is_unique,
           status['sirius_feature_id'].nunique(), len(status))
    add_qc(f'{mode}_successful_rows_finite',
           bool(np.isfinite(matrix[success_mask]).all()) if success_mask.any() else False,
           int(success_mask.sum()), '>0')
    add_qc(f'{mode}_successful_rows_in_0_1',
           bool(((matrix[success_mask] >= 0) & (matrix[success_mask] <= 1)).all()) if success_mask.any() else False,
           int(success_mask.sum()), '>0')
    add_qc(f'{mode}_fingerprint_coverage', success_mask.all(),
           f'{int(success_mask.sum())}/{len(status)}', '100%', severity='WARN')

shared = alignment['_merge'].eq('both')
add_qc('cross_mode_shared_description_consistent',
       bool(alignment.loc[shared, 'description_consistent'].all()),
       int(alignment.loc[shared, 'description_consistent'].sum()), int(shared.sum()))
add_qc('cross_mode_shared_type_consistent',
       bool(alignment.loc[shared, 'type_consistent'].all()),
       int(alignment.loc[shared, 'type_consistent'].sum()), int(shared.sum()))

qc = pd.DataFrame(qc_rows)
qc_path = OUTPUT_DIR / 'development_sirius_result_qc_checks.csv'
qc.to_csv(qc_path, index=False, encoding='utf-8-sig')
display(qc)
assert not qc['status'].eq('FAIL').any(), qc.loc[qc['status'].eq('FAIL')]
print('硬性 QC：PASS；WARN 表示需要在论文中报告的指纹缺失/覆盖率，而不是数据守恒失败。')

In [ ]:
# 保存可复现运行清单和所有正式输出的 SHA256。
# Windows下，正在被SIRIUS打开的.sirius数据库可能被独占锁定。锁定时先记录文件身份，
# 不让清单生成失败；关闭SIRIUS后可单独重新运行本单元补齐项目SHA256。
def file_provenance(path):
    path = Path(path)
    stat = path.stat()
    record = {
        'path': str(path.resolve()),
        'size_bytes': int(stat.st_size),
        'last_write_time_utc': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
        'sha256': None,
        'hash_status': None,
    }
    try:
        record['sha256'] = sha256_file(path)
        record['hash_status'] = 'COMPLETE'
    except PermissionError:
        record['hash_status'] = 'PENDING_FILE_LOCKED_BY_SIRIUS'
    return record

formal_outputs = [
    OUTPUT_DIR / 'fingerprint_dictionary_positive.csv',
    OUTPUT_DIR / 'fingerprint_dictionary_negative.csv',
    OUTPUT_DIR / 'positive_fingerprint_probabilities.npy',
    OUTPUT_DIR / 'negative_fingerprint_probabilities.npy',
    OUTPUT_DIR / 'positive_fingerprint_rows.csv.gz',
    OUTPUT_DIR / 'negative_fingerprint_rows.csv.gz',
    OUTPUT_DIR / 'fingerprint_cross_mode_alignment.csv.gz',
    OUTPUT_DIR / 'development_sirius_result_qc_checks.csv',
]

input_file_provenance = {
    'feature_manifest': file_provenance(FEATURE_MANIFEST_PATH),
    'uniform_retry_audit': file_provenance(RETRY_AUDIT_PATH),
    'positive_sirius_project': file_provenance(PROJECTS['positive']['project_path']),
    'negative_sirius_project': file_provenance(PROJECTS['negative']['project_path']),
}

run_manifest = {
    'export_version': 'SIRIUS_DEVELOPMENT_FINGERPRINT_EXPORT_V1',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'scientific_scope': 'Development public spectra only; rank-1 SIRIUS formula; no structure DB; ChemVista locked',
    'sirius_info': SIRIUS_INFO,
    'api_base_redacted': 'http://127.0.0.1:<dynamic-port>',
    'selection_rule': {
        'formula': 'SIRIUS molecular formula candidate with rank == 1',
        'fallback_to_lower_rank_formula': False,
        'fingerprint_semantics': 'CSI:FingerID probability vector ordered by charge-specific relativeIndex',
        'cross_mode_alignment_key': 'absoluteIndex',
        'post_retry_404_policy': 'fingerprint_unavailable_after_retry; retained as coverage limitation',
    },
    'uniform_retry_audit': RETRY_AUDIT,
    'counts': {},
    'input_hashes': {key: value['sha256'] for key, value in input_file_provenance.items()},
    'input_file_provenance': input_file_provenance,
    'project_hashes_complete': all(
        input_file_provenance[key]['hash_status'] == 'COMPLETE'
        for key in ('positive_sirius_project', 'negative_sirius_project')
    ),
    'output_hashes': {},
}

for mode in ('positive', 'negative'):
    status = STATUS_TABLES[mode]
    run_manifest['counts'][mode] = {
        'features': int(len(status)),
        'fingerprint_dimension': int(MATRICES[mode].shape[1]),
        'success': int(status['status'].eq('success').sum()),
        'failed_or_missing': int((~status['status'].eq('success')).sum()),
        'status_counts': {str(k): int(v) for k, v in status['status'].value_counts(dropna=False).items()},
    }

for path in formal_outputs:
    run_manifest['output_hashes'][path.name] = sha256_file(path)

manifest_path = OUTPUT_DIR / 'development_sirius_result_manifest.json'
manifest_path.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(run_manifest['counts'], ensure_ascii=False, indent=2))
print('manifest:', manifest_path)
pending_hashes = [
    key for key, value in input_file_provenance.items()
    if value['hash_status'] != 'COMPLETE'
]
if pending_hashes:
    print('提示：以下文件正在被SIRIUS锁定，清单已生成但SHA256待补：', pending_hashes)
    print('关闭SIRIUS后，只需重新运行本单元，即可补齐项目SHA256。')
else:
    print('全部输入和输出SHA256：COMPLETE')

## 输出解释与下一步

- 负、正模式模块分别运行并立即落盘，不要求两个SIRIUS项目同时打开。
- `fingerprint_dictionary_*.csv`：每一位的 `relativeIndex → absoluteIndex → description/Type/F1` 映射。
- `*_fingerprint_probabilities.npy`：`float32`概率矩阵；失败feature对应整行 `NaN`。
- `*_fingerprint_rows.csv.gz`：矩阵行号、匿名feature ID、最佳分子式、分数、成功/失败原因。
- `fingerprint_cross_mode_alignment.csv.gz`：正负模式按 `absoluteIndex` 的定义对照。
- `development_sirius_result_qc_checks.csv`：硬性QC与覆盖率警告。
- `development_sirius_result_manifest.json`：版本、选择规则、数量和SHA256。

下一步才是把成功指纹按 `development_sirius_feature_manifest.csv.gz` 映射至化合物与冻结fold，预先固定正负模式聚合规则，然后训练SIRIUS-PBT预测头。ChemVista仍保持锁定。